In [0]:
import pandas as pd

pdf = spark.table("projeto_churn.silver.clientes").toPandas()

colunas_features = [
    "tenure", "MonthlyCharges", "TotalCharges", "Contract", "InternetService", 
    "PaymentMethod", "TechSupport", "OnlineSecurity", "PaperlessBilling" 
]

X = pd.get_dummies(pdf[colunas_features], drop_first=True)
y = pdf["churn_flag"]

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

with mlflow.start_run(run_name="random_forest_v1"):
    modelo = RandomForestClassifier(
        n_estimators=200, max_depth=8,
        class_weight="balanced", random_state=42
    )
    modelo.fit(X_train, y_train)

    probs = modelo.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)

    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 8)
    mlflow.log_metric("auc", auc)
    mlflow.sklearn.log_model(modelo, "modelo")

    print(f"AUC: {auc:.4f}")
    print(classification_report(y_test, modelo.predict(X_test)))

2026/09/17 00:21:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-3174491f-34d5.cloud.databricks.com/ml/experiments/2087537773859693/models/m-12d273668a5a4284943bc76a0bac0032?o=7474660421217040
2026/09/17 00:21:43 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.16.0/ml/model/signatures.html for instructions on setting signature on models.


AUC: 0.8363
              precision    recall  f1-score   support

           0       0.90      0.75      0.82      1035
           1       0.53      0.78      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409



In [0]:
# 1. pede para o modelo prever a probabilidade de churn de toda a base
pdf["score_churn"] = modelo.predict_proba(X)[:, 1]

# 2. cria um DataFrame focado no negócio, juntando o risco com as informações financeiras
df_scores = spark.createDataFrame(
    pdf[["customerID", "score_churn", "MonthlyCharges", "Contract", "churn_flag"]]
)

# 3. salva o resultado de volta no banco de dados como uma tabela gold
(df_scores.write
    .format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("projeto_churn.gold.scores_risco")
)

<h3> A Pergunta de negócio final    

In [0]:
%sql
SELECT 
    customerID,
    Contract,
    ROUND(score_churn, 3) AS risco,
    MonthlyCharges AS receita_mensal_em_risco
FROM projeto_churn.gold.scores_risco
WHERE score_churn > 0.6
ORDER BY MonthlyCharges DESC
LIMIT 50;

customerID,Contract,risco,receita_mensal_em_risco
7279-BUYWN,Month-to-month,0.631,113.2
1583-IHQZE,Month-to-month,0.795,112.95
2403-BCASL,One year,0.651,111.95
3292-PBZEJ,Month-to-month,0.709,111.4
0115-TFERT,Month-to-month,0.715,111.2
6030-REHUX,Month-to-month,0.656,110.85
3336-JORSO,Month-to-month,0.691,110.45
9851-KIELU,Month-to-month,0.746,110.1
3174-RKMOW,Month-to-month,0.678,109.95
3992-YWPKO,Month-to-month,0.792,109.9
